In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

In [40]:
data = pd.read_csv("Data/USO/USO_stats.csv", index_col=0)
data = data.dropna()

In [41]:
data

,time_gap,internal_return,cc_return,hour,hourly_typical_volume,relative_volume,volatility,target
27,1.0,0.000593,-0.000148,13,2.498750e+04,0.160580,0.003402,1.0
28,1.0,-0.000592,0.000296,14,8.900833e+04,0.093671,0.003271,1.0
29,1.0,0.004741,0.003848,15,4.658333e+04,0.068157,0.003583,1.0
30,1.0,0.005751,0.005751,16,4.113333e+04,0.106058,0.003950,1.0
31,1.0,0.001907,0.001319,17,3.177500e+04,0.228954,0.003064,1.0
...,...,...,...,...,...,...,...,...
72223,1.0,-0.011576,-0.012120,18,7.731858e+05,0.524129,0.005213,1.0
72224,1.0,0.003904,0.003904,19,1.121144e+06,0.346757,0.005505,1.0
72225,1.0,0.000347,0.000208,20,6.356008e+04,1.156112,0.005495,1.0
72226,1.0,-0.000069,0.000139,21,2.430599e+04,1.622659,0.005497,0.0


**Training**

In [42]:
split_id = int(len(data) * 0.8)
X = data.drop(columns = "target")
y = data["target"]

In [43]:
X_train = X.iloc[:split_id]
X_test = X.iloc[split_id:]
y_train = y.iloc[:split_id]
y_test = y.iloc[split_id:]

In [44]:
clf = tree.DecisionTreeClassifier(random_state=42)

path = clf.cost_complexity_pruning_path(X_train, y_train)

In [45]:
param_grid = {'ccp_alpha': np.unique(path['ccp_alphas'])}

In [46]:
tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
clf_grid = GridSearchCV(clf, param_grid, cv = tscv, scoring = 'accuracy', n_jobs = -1)
clf_grid.fit(X_train, y_train)

In [ ]:
print("Best ccp_alpha:", clf_grid.best_params_["ccp_alpha"])
print("Best CV accuracy:", clf_grid.best_score_)

In [ ]:
best_tree = clf_grid.best_estimator_

In [ ]:
print("Test accuracy:", best_tree.score(X_test, y_test))
print("Tree depth:", best_tree.get_depth())
print("Number of leaves:", best_tree.get_n_leaves())

In [ ]:
import joblib
joblib.dump(clf_grid, "Models/tree_grid_search.joblib")
joblib.dump(best_tree, "Models/best_tree.joblib")